# MediPilot — ASR bake-off (Kaggle T4)

**What this decides:** which Whisper the intake conversation should run on, and what the
cheap options actually cost in accuracy.

`round2-implementation-plan.html` §15 commits to *"Whisper-class ASR, quantised, evaluated on
Indian-accented and code-mixed speech before we depend on it"*, and §16 lists speech integration
as a Medium risk. This notebook is the evidence for both. It produces a table you can paste
straight into the report.

**Why Kaggle:** the development laptop has an AMD Radeon RX 6500M — no CUDA, no ROCm on Windows —
so `torch` is CPU-only there and `large-v3` is unusable. Kaggle's T4 makes the comparison possible
at all.

---

## Setup — do these three things first

1. **Settings → Accelerator → GPU T4 x2.** (One T4 is enough; the second is free.)
2. **Upload the repo as a Kaggle Dataset.** Zip `medipilot-model/` (or at minimum the `eval/`,
   `speech/` and `intake/` folders — the audio fixtures live in `speech/`) and add it to this
   notebook via *+ Add Input → Datasets → Upload*. The next cell finds it wherever it lands.
3. *(Optional, for the hosted comparison)* **Add-ons → Secrets → `GROQ_API_KEY`.**

## How to read the result

Backends are ranked by **silence hallucinations first**, then WER. A model that emits
*"Thank you."* over a silent recording is rejected however good its WER is — in this pipeline a
hallucinated sentence becomes the patient's own reported symptoms and is fed to the red-flag table.

## 1 · Environment check

T4s are Turing: they do **not** support bfloat16. Everything below uses float16 or int8. If a
model card tells you to use bf16, that is a T4 crash waiting to happen.

In [ ]:
import subprocess, torch, platform
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version',
                      '--format=csv'], capture_output=True, text=True).stdout)
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('bf16 supported:', torch.cuda.is_bf16_supported(), '(False on T4 — use fp16)')
print('python', platform.python_version())

## 2 · Install

`faster-whisper` is included because §15 says *quantised*: it runs the same weights through
CTranslate2 with int8, which is the only configuration that could ever run on the demo laptop's
CPU. Measuring what that costs is the point.

In [ ]:
%pip install -q openai-whisper faster-whisper groq soundfile 2>&1 | tail -2
print('installed')

## 3 · Find the repo

Looks for the uploaded dataset under `/kaggle/input`, falls back to the working directory so the
same notebook runs locally without edits.

In [ ]:
import os, sys, shutil

def find_repo():
    # The repo root is whichever directory contains BOTH eval/ and speech/.
    # A shallow glob is not enough: Kaggle sometimes mounts a dataset at
    # /kaggle/input/datasets/<owner>/<slug>/... (one level deeper than the
    # classic /kaggle/input/<slug>/...) depending on how it was attached.
    # os.walk with a depth cap is robust to either layout.
    for search_root in ('/kaggle/input', '/kaggle/working', '.', '..'):
        if not os.path.isdir(search_root):
            continue
        for dirpath, dirnames, _ in os.walk(search_root):
            rel = os.path.relpath(dirpath, search_root)
            depth = 0 if rel == '.' else rel.count(os.sep) + 1
            if depth > 6:
                dirnames[:] = []  # do not descend further from here
                continue
            if 'eval' in dirnames and 'speech' in dirnames:
                return os.path.abspath(dirpath)
    raise SystemExit(
        'Could not find the repo. Upload medipilot-model/ as a Kaggle Dataset '
        '(it must contain eval/ and speech/) and add it via + Add Input.')

REPO = find_repo()
sys.path.insert(0, REPO)
print('repo:', REPO)

# /kaggle/input is read-only, and the runner writes result files. Copy to a
# writable location so nothing later fails on a permission error.
if REPO.startswith('/kaggle/input'):
    dst = '/kaggle/working/medipilot-model'
    if not os.path.exists(dst):
        shutil.copytree(REPO, dst)
    REPO = dst
    sys.path.insert(0, REPO)
    print('working copy:', REPO)
os.chdir(REPO)

In [ ]:
from eval import metrics
from eval.run_asr_bakeoff import (
    load_manifest, run_backend, write_reference_template,
    LOCAL_WHISPER_SIZES, GROQ_ASR_MODELS, SUMMARY_COLUMNS,
)

cases = load_manifest()
print(f'{len(cases)} fixtures')
for c in cases:
    ok = os.path.exists(os.path.join(REPO, c['path']))
    ref = len(c.get('references', []))
    print(f"  {c['id']:<12} {c['path']:<32} {'OK' if ok else 'MISSING':<8} "
          f"{ref} reference(s){'  <-- needs a hand-written transcript' if not ref and c.get('kind')!='silence' else ''}")

## 4 · Local Whisper, every size

Models are loaded one at a time and released before the next, so six sizes fit in 16 GB.
`large-v3` takes a few minutes to download the first time.

The `tiny`/`base`/`small` rows are not filler — they are the only sizes that could run on the
demo laptop's CPU if the hosted path were ever unavailable, so their WER is the price of that
fallback.

In [ ]:
import gc, time, torch
from speech.whisper_stt import WhisperSTT

SIZES = ['tiny', 'base', 'small', 'medium', 'turbo', 'large-v3']
summaries = []

for size in SIZES:
    print(f'\n=== whisper-local:{size} ===')
    t0 = time.perf_counter()
    stt = WhisperSTT(size)
    print(f'  loaded in {time.perf_counter()-t0:.1f}s')
    try:
        s = run_backend(f'whisper-local:{size}', lambda p: stt.transcribe(p),
                        cases, verbose=True)
        s['vram_peak_gb'] = round(torch.cuda.max_memory_allocated()/1e9, 2)
        summaries.append(s)
    finally:
        del stt
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

## 5 · Quantised — faster-whisper (CTranslate2)

Same weights, int8 and float16 compute. This is the §15 *quantised* claim under test: how much
accuracy does int8 cost, and how much latency does it buy?

The adapter below routes through `asr_common.build_result` so these rows carry the identical
reliability flags and are directly comparable to the rows above — the comparison measures the
models, not two different post-processors.

In [ ]:
from faster_whisper import WhisperModel
from speech import asr_common

def faster_whisper_backend(size, compute_type):
    model = WhisperModel(size, device='cuda', compute_type=compute_type)

    def _transcribe(path):
        # The energy gate must run here too, or these rows get an unearned
        # advantage on the silence fixtures.
        import soundfile as sf, numpy as np
        wav, sr = sf.read(path, dtype='float32', always_2d=True)
        mono = wav.mean(axis=1)
        if asr_common.is_effectively_silent(mono):
            return asr_common.empty_result(True, f'faster-whisper:{size}/{compute_type}')
        segs, info = model.transcribe(path, task='transcribe', language=None,
                                      condition_on_previous_text=False, temperature=0)
        segs = list(segs)
        return asr_common.build_result(
            text=''.join(s.text for s in segs),
            language=info.language,
            segments=[{'no_speech_prob': s.no_speech_prob,
                       'avg_logprob': s.avg_logprob,
                       'compression_ratio': s.compression_ratio} for s in segs],
            backend=f'faster-whisper:{size}/{compute_type}',
            language_confidence=info.language_probability,
        )
    return model, _transcribe

for size, ct in [('small', 'int8'), ('large-v3', 'int8_float16'), ('large-v3', 'float16')]:
    name = f'faster-whisper:{size}/{ct}'
    print(f'\n=== {name} ===')
    model, fn = faster_whisper_backend(size, ct)
    try:
        summaries.append(run_backend(name, fn, cases, verbose=True))
    finally:
        del model
        gc.collect(); torch.cuda.empty_cache()

## 6 · Indic-tuned Whisper

`vasista22/whisper-hindi-large-v2` is fine-tuned on Hindi and is the obvious challenger for the
Hinglish fixture, which is the case the talk-first design (§01 D12) actually rests on.

Skip this cell if you are short on time — it is a comparison, not a dependency.

In [ ]:
from transformers import pipeline
import numpy as np

INDIC_MODELS = ['vasista22/whisper-hindi-large-v2']


def _load_16k(path):
    """Decode to mono float32 @16kHz ourselves and hand the pipeline a
    raw array rather than a file path. Several .ogg fixtures raise a
    'num_frames' KeyError when this pipeline's own path-based audio
    loader tries to read them (a known quirk of this pipeline/format
    combination, not the audio) -- pre-decoding sidesteps it entirely.
    """
    import soundfile as sf

    wav, sr = sf.read(path, dtype='float32', always_2d=True)
    mono = wav.mean(axis=1)
    if sr != 16000 and len(mono) > 1:
        idx = np.linspace(0, len(mono) - 1, int(len(mono) * 16000 / sr))
        mono = np.interp(idx, np.arange(len(mono)), mono).astype(np.float32)
    return mono


for model_id in INDIC_MODELS:
    print(f'\n=== {model_id} ===')
    try:
        pipe = pipeline('automatic-speech-recognition', model=model_id,
                        device=0, torch_dtype=torch.float16)

        def _transcribe(path, pipe=pipe, model_id=model_id):
            mono = _load_16k(path)
            if asr_common.is_effectively_silent(mono):
                return asr_common.empty_result(True, model_id)
            out = pipe({'array': mono, 'sampling_rate': 16000})
            # These checkpoints are single-language by construction, so the
            # language is asserted rather than detected. Recorded as such —
            # it means the language_accuracy column is not meaningful for
            # this row, which is itself a finding: a Hindi-only model cannot
            # tell you it heard English.
            return asr_common.build_result(out['text'], 'hi', [], model_id)

        summaries.append(run_backend(model_id, _transcribe, cases, verbose=True))
    except Exception as e:
        print('  skipped:', e)
    finally:
        gc.collect(); torch.cuda.empty_cache()

## 7 · Hosted comparison — Groq

This is the backend the demo actually serves from (`speech/groq_asr.py`), so its row is the one
that has to be defensible. Needs the `GROQ_API_KEY` Kaggle Secret.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['GROQ_API_KEY'] = UserSecretsClient().get_secret('GROQ_API_KEY')
    print('GROQ_API_KEY loaded from Kaggle Secrets')
except Exception as e:
    print('No Groq secret — skipping hosted backends.', e)

if os.environ.get('GROQ_API_KEY'):
    from speech.groq_asr import GroqWhisperSTT
    for model in GROQ_ASR_MODELS:
        print(f'\n=== groq:{model} ===')
        stt = GroqWhisperSTT(model)
        summaries.append(run_backend(f'groq:{model}', stt.transcribe, cases, verbose=True))

## 8 · Results

The table below is the deliverable. Paste it into the report.

In [ ]:
summaries.sort(key=lambda s: (s['silence_hallucinations'],
                              s['mean_wer'] if s['mean_wer'] is not None else 1e9))

cols = ['backend', 'mean_wer', 'median_wer', 'mean_cer', 'language_accuracy',
        'silence_hallucinations', 'n_speech', 'mean_latency_s']
if any('vram_peak_gb' in s for s in summaries):
    cols.append('vram_peak_gb')

table = metrics.to_markdown_table(summaries, cols)
print('## ASR bake-off\n')
print(table)
print('\nRanked by silence hallucinations first, then mean WER.')

from IPython.display import Markdown, display
display(Markdown(table))

### Per-fixture detail

Where the averages come from. The Hinglish and second-speaker rows are the ones worth reading
closely — an average over eight fixtures hides a lot.

In [ ]:
import pandas as pd

detail = [{'backend': s['backend'], **{k: v for k, v in r.items() if not k.startswith('_')}}
          for s in summaries for r in s['_rows']]
df = pd.DataFrame(detail)
pd.set_option('display.max_colwidth', 90)
display(df[['backend', 'id', 'wer', 'cer', 'detected_language', 'hypothesis']])

## 9 · Save

Both files land in `/kaggle/working` and can be downloaded from the notebook's Output tab.

`asr_reference_template.json` lists the fixtures that still owe a hand-written transcript
(`F-01`..`F-03`, `HINGLISH-01`). **Listen to each one and write the reference yourself.** Promoting
a model's own output to ground truth scores that model against itself and reports a free 0.000
WER, which would make this entire notebook decorative.

In [ ]:
import json

with open('/kaggle/working/asr_bakeoff_results.json', 'w', encoding='utf-8') as fh:
    json.dump(summaries, fh, indent=2, ensure_ascii=False)

with open('/kaggle/working/asr_bakeoff_table.md', 'w', encoding='utf-8') as fh:
    fh.write('## ASR bake-off\n\n' + table + '\n')

write_reference_template(summaries, '/kaggle/working/asr_reference_template.json')
print('saved to /kaggle/working/')